In [3]:
import numpy as np
from scipy.fftpack import fft, rfftfreq, rfft
import matplotlib.pyplot as plt

In [46]:
import numpy as np
from scipy.fft import rfft, rfftfreq
import plotly.graph_objects as go
from plotly.subplots import make_subplots

bandwidth = 15_000
signal_space = np.linspace(0, 1, 1_000_000)

sigma_t = 1 / (2 * np.pi * bandwidth) * 10
t_center = 0.5
gaussian_envelope = np.exp(-(signal_space - t_center)**2 / (2 * sigma_t**2))

signal_l_frequency = 10_000
signal_l = gaussian_envelope * np.sin(2 * np.pi * signal_l_frequency * signal_space)

signal_r_frequency = 3000
signal_r = 2/3 * gaussian_envelope * np.sin(2 * np.pi * signal_r_frequency * signal_space)

signal_plus = signal_l + signal_r
signal_minus = signal_l - signal_r
pilot_frequency = 18_000
pilot_signal = np.sin(2 * np.pi * pilot_frequency * signal_space)
shift_frequency = pilot_frequency * 2
shift_signal = np.cos(2 * np.pi * shift_frequency * signal_space)
shifted_signal = signal_minus * shift_signal
signal = signal_plus + shifted_signal + pilot_signal / 3_000

# FFTs
fft_freq = rfftfreq(signal.size, d=signal_space[1] - signal_space[0])
fft_full = np.abs(rfft(signal))
fft_l    = np.abs(rfft(signal_l))
fft_r    = np.abs(rfft(signal_r))

freq_mask = fft_freq <= 55_000
time_mask = signal_space <= 0.001
fmax = np.max(fft_full[freq_mask])

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=("Signal (first 1ms)", "FFT of the Signal"),
    vertical_spacing=0.12
)

# --- Time domain ---
fig.add_trace(go.Scatter(
    x=signal_space[time_mask],
    y=signal[time_mask],
    mode="lines",
    name="Signal",
    line=dict(width=2)
), row=1, col=1)

# --- Shaded regions ---
fig.add_vrect(x0=0, x1=bandwidth,
    fillcolor="red", opacity=0.15, line_width=0, row=2, col=1)
fig.add_vrect(x0=shift_frequency - bandwidth, x1=shift_frequency,
    fillcolor="blue", opacity=0.15, line_width=0, row=2, col=1)
fig.add_vrect(x0=shift_frequency, x1=shift_frequency + bandwidth,
    fillcolor="green", opacity=0.15, line_width=0, row=2, col=1)

# --- Vertical line at 2*pilot via shape (reliable row targeting) ---
fig.add_shape(
    type="line",
    x0=shift_frequency, x1=shift_frequency,
    y0=0, y1=fmax,
    line=dict(color="red", dash="dash", width=2),
    row=2, col=1
)

# --- FFT traces ---
fig.add_trace(go.Scatter(
    x=fft_freq[freq_mask],
    y=fft_full[freq_mask],
    mode="lines",
    name="Combined",
    line=dict(width=2),
), row=2, col=1)

fig.add_trace(go.Scatter(
    x=fft_freq[freq_mask],
    y=fft_l[freq_mask],
    mode="lines",
    name=f"Left ({signal_l_frequency/1000:.0f} kHz)",
    line=dict(color="cyan", width=2.5, dash="dot"),
), row=2, col=1)

fig.add_trace(go.Scatter(
    x=fft_freq[freq_mask],
    y=fft_r[freq_mask],
    mode="lines",
    name=f"Right ({signal_r_frequency/1000:.0f} kHz)",
    line=dict(color="green", width=2.5, dash="dot"),
), row=2, col=1)

fig.update_layout(
    height=700,
    legend=dict(orientation="h", y=-0.12),
)
fig.update_xaxes(title_text="Time [s]", row=1, col=1)
fig.update_yaxes(title_text="Amplitude", row=1, col=1)
fig.update_xaxes(title_text="Frequency [Hz]", row=2, col=1)
fig.update_yaxes(title_text="Magnitude", row=2, col=1)

fig.show()

# Pipeline

Input stereo signal is 20Hz-20kHz

Sample at 1Mhz and sample down to 50kHz with FIR etc

Then signal processing, upsampling etc (see above)

calc back for 33Mhz driving and do proportional FM transforms, then the third harmonic will have 100Mhz approx.

Consider if simple jerryrigged analog LC filter at output pin between antenna

22 pF + small trimmer cap (5–20 pF) in parallel with a 100 nH inductor

FPGA pin ---[L]---[C]--- wire antenna

                  |

                 GND  (antenna ground/return)

- ADC x2 einlesen
- CIC downsampling x10
- FIR for CIC correction and targed 20Hz-15kHz band
- subtract and add lines l+r l-r
- generate omega_1, omega_2 shift pilot/shift signal (DDS + cordic)
- shift to 2x pilot with x sin(omega_2)
- add both signals (n bit f_s)
- dds for output signal
- thresholding for square wave output (half amplitude)
- make antenna (with mayb bandpass filter)
- V make presentation 
- D make figures


COMPONENTS TO CODE:

- V ADC + CIC + FIR
- V DDS (overflow triangle thingy)
- D CORDIC for sine from dds -------------------- DONE
- D THRESHOLD for square from triangle ---------- 
- D STATE MACHINE/PIPELINE for R-L transforms --- DONE
- V PROJECT FRAMEWORK


In [ ]:
input_band = [20, 20_000] #analog audio signal
input_channels = 2

adc_sample_rate = 1_000_000
adc_bit_depth = 16
downsampling_factor = 20
fir_filter_order = 64

system_clock = 100_000_000

output_drive_clock = 33_333_333